# IFCNetCore — DuoDuoCLIP visual embeddings

Runs DuoDuoCLIP (`Four_1to6F_bs1600_LT6.ckpt`) over all 12 stock renders per
object. **Unlike SigLIP / DINOv3, `encode_image` accepts all 12 views at once
and pools internally** — no extra mean-pool afterwards.

**Upload to your Drive:** `IFCNetCorePng.zip` (445 MB renders zip).
**Output to Drive:** `duoduo_ifcnet_colorless.zip`.

Resume-safe: re-running skips obj_ids whose `.npy` already exists.

⚠️ Use a **fresh runtime** — DuoDuoCLIP installs its own forked `open_clip_mod`
which can shadow other packages.

Pick a GPU runtime first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
from pathlib import Path

DRIVE_ZIP    = Path('/content/drive/MyDrive/IFCNetCorePng.zip')
DRIVE_OUT    = Path('/content/drive/MyDrive')

RENDERS_ROOT  = Path('/content/data/IFCNetCore/renders')
FEATURES_ROOT = Path('/content/data/IFCNetCore/processed/rendered_features')
METADATA_PATH = Path('/content/data/IFCNetCore/metadata.json')

assert DRIVE_ZIP.exists(), f'Upload IFCNetCorePng.zip to {DRIVE_ZIP.parent} first.'
print('zip   :', DRIVE_ZIP, '(', DRIVE_ZIP.stat().st_size // (1024*1024), 'MB)')
print('drive :', DRIVE_OUT)

In [ ]:
RENDERS_ROOT.mkdir(parents=True, exist_ok=True)
!unzip -q -n "$DRIVE_ZIP" -d "$RENDERS_ROOT"
n_png = sum(1 for _ in RENDERS_ROOT.rglob('*.png'))
print(f'unzipped: {n_png} PNGs')

In [ ]:
import json
rows = []
for class_dir in sorted(RENDERS_ROOT.iterdir()):
    if not class_dir.is_dir(): continue
    for split_dir in sorted(class_dir.iterdir()):
        if not split_dir.is_dir(): continue
        obj_ids = sorted({p.stem.rsplit('.', 1)[0] for p in split_dir.glob('*.png')})
        for oid in obj_ids:
            views = sorted(split_dir.glob(f'{oid}.*.png'))
            rows.append({'obj_id': oid, 'ifc_class': class_dir.name,
                         'split': split_dir.name, 'num_renders': len(views)})
METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
METADATA_PATH.write_text(json.dumps(rows, indent=2))
print(f'{len(rows)} objects, {len({r["ifc_class"] for r in rows})} classes')

In [ ]:
import json, shutil, subprocess
from dataclasses import dataclass
from tqdm import tqdm
import numpy as np


@dataclass(frozen=True)
class ObjectEntry:
    obj_id: str
    ifc_class: str
    split: str
    views: list


def load_entries():
    out = []
    for r in json.loads(METADATA_PATH.read_text()):
        d = RENDERS_ROOT / r['ifc_class'] / r['split']
        out.append(ObjectEntry(r['obj_id'], r['ifc_class'], r['split'],
                               sorted(d.glob(f"{r['obj_id']}.*.png"))))
    return out


def filter_pending(entries, out_dir):
    return [e for e in entries if not (out_dir / f'{e.obj_id}.npy').exists()]


def zip_and_upload(features_subdir, archive_name, drive_out=DRIVE_OUT):
    src = features_subdir / 'colorless'
    n = sum(1 for _ in src.glob('*.npy'))
    if n == 0:
        print(f'  [warn] {src} empty — nothing to zip'); return None
    local = Path('/content') / archive_name
    if local.exists(): local.unlink()
    subprocess.run(['zip', '-qr', str(local), 'colorless'], cwd=str(features_subdir), check=True)
    drive_out.mkdir(parents=True, exist_ok=True)
    drive_zip = drive_out / archive_name
    shutil.copy2(local, drive_zip)
    print(f'  ✓ zipped {n} files → {drive_zip}  ({drive_zip.stat().st_size/1024/1024:.1f} MB)')
    return drive_zip


ENTRIES = load_entries()
print(f'{len(ENTRIES)} object entries')

## Clone DuoDuoCLIP + install its forked open_clip

This step rewrites `open_clip` to DuoDuo's fork — that's why this notebook is
isolated from the SigLIP / DINOv3 ones.

In [ ]:
if not Path('/content/DuoduoCLIP').exists():
    !git clone -q https://github.com/3dlg-hcvc/DuoduoCLIP.git /content/DuoduoCLIP
!pip install -q /content/DuoduoCLIP/open_clip_mod/

## Run DuoDuoCLIP (~1.5 h on T4 for 7930 objects)

In [ ]:
import sys, torch
from PIL import Image
sys.path.insert(0, '/content/DuoduoCLIP')
from src.model.wrapper import get_model

DUODUO_OUT = FEATURES_ROOT / 'duoduo' / 'colorless'
DUODUO_OUT.mkdir(parents=True, exist_ok=True)
DUODUO_CKPT = 'Four_1to6F_bs1600_LT6.ckpt'
DUODUO_RES = 224

pending = filter_pending(ENTRIES, DUODUO_OUT)
print(f'[duoduo] {len(pending)} / {len(ENTRIES)} pending')

if pending:
    duoduo = get_model(DUODUO_CKPT, device='cuda')

    # smoke
    with torch.no_grad():
        _pil = Image.open(pending[0].views[0]).convert('RGB').resize((DUODUO_RES, DUODUO_RES))
        _arr = np.expand_dims(np.asarray(_pil), 0)
        _out = duoduo.encode_image(_arr)
    print(f'[duoduo] smoke: out type={type(_out).__name__}  shape={tuple(_out.shape)}')

    for e in tqdm(pending, desc='duoduo'):
        try:
            imgs = []
            for v in e.views:
                pil = Image.open(v).convert('RGB').resize((DUODUO_RES, DUODUO_RES))
                imgs.append(np.expand_dims(np.asarray(pil), 0))
            batch = np.concatenate(imgs, axis=0)
            with torch.no_grad():
                feats = duoduo.encode_image(batch)        # multi-view pooled internally
            vec = feats.cpu().numpy().reshape(-1).astype(np.float32)
            np.save(DUODUO_OUT / f'{e.obj_id}.npy', vec)
        except Exception as exc:
            print(f'  [error] {e.obj_id}: {exc}')

    del duoduo
    torch.cuda.empty_cache()

print('done:', len(list(DUODUO_OUT.glob('*.npy'))), 'embeddings written')

In [ ]:
zip_and_upload(FEATURES_ROOT / 'duoduo',
               archive_name='duoduo_ifcnet_colorless.zip')